# Does the fine-tuned model itself use the subcomponents Γ switches on at the trigger?

*E38, `experiments/recruitment_test`, 2026-10-27. Record, with every number and the full scoring:
`reports/analyses/2026-10-27_analysis_recruitment-test.md`.*

## The answer

Partly, and not as filed. Removing the subcomponents Γ switches on from the fine-tuned weights takes away a third to a
half of the fine-tune's gain on two of three tasks. That is far more than the average random set of the same size. But
removing them from the original model, which we predicted would do nothing, already takes away a sixth to a quarter.
They push toward the target before any fine-tuning. How much of the fine-tune runs through them is not yet measured.

## What we were trying to test

**The question.** After a narrow fine-tune, Γ stays frozen and reads the fine-tuned model's activations. At the trigger
position it then switches on a set of the original model's subcomponents. We call it the *recruited set* S: the alive
subcomponents whose mean causal importance at the trigger rises by more than 0.3. Until now this set was only read in
the CI-masked model, old subcomponents under new importance values. The fine-tuned model itself has no Γ. So we asked
it directly: does it act through S?

**The states.** Three tasks, each a 10-step Adam fine-tune of all 24 decomposed matrices: emoticon, where a `:)`-style
trigger is followed by `o`; ` which` followed by ` was`; and ` we` followed by ` just`. Two seeds per task. A fine-tune of
the same size to ` http` at the same triggers is the null.

**The measurement.** *Removing* a set means subtracting each of its rank-1 matrices $u_c v_c^\top$ from the real
weights, at every position, with no masks. We read the target's mean log-odds at the trigger on held-out windows. The
*removed share* is $\rho = (T_1 - T_1^{\text{without set}})/G$, with $T_1$ the fine-tuned log-odds and $G$ the
fine-tune's gain over the original model. A share of 0 means removing the set does nothing; 1 means it undoes the
fine-tune.

The share is in log-odds at near-certainty, so it overstates the effect in probability. On emoticon a share of 0.5
still leaves the target the likely next token.

**What each comparison rules out.**

- *Twenty random sets* of the same size per matrix: that removing any that many subcomponents lowers the target.
- *The null's recruited set*: that any switched-on set carries the target.
- *The busy set*, the most important subcomponents at the trigger in the original model: a check that removal bites.
- *S removed from the original weights*: we expected nothing, because S is the set the fine-tune switches on.

Two descriptive stages followed. The *input trace* reads the cosine between the fine-tune's mean change in each
matrix's input at the trigger and the target's *natural direction*. That direction is the mean input at Pile positions
just before the target minus the mean input everywhere. The *census* lists where the original model switches each
member of S on in Pile text, and which tokens follow there.

## What we expected, and why

- **Removing S undoes at least half the gain** on two of three tasks, with random sets and the null's set near zero.
  Medium-low confidence: in the masked model S carried only about half the switching effect.
- **Removing S is specific**, costing at most three times the fine-tune's own damage away from the trigger.
- **S is inert in the original model**, high confidence. It is defined by the fine-tune switching it on.
- **The update points at the target's contexts:** in the layer holding most of S, cosines of at least 0.3.
- **The census finds the target's machinery in S**, as an earlier overlap measure suggested.

## What we observed

The apparatus reproduced the earlier run that defined S on its second attempt. The first attempt stopped because the
fine-tuned log-odds on emoticon sat about a tenth of a nat from the earlier run's, against a tolerance of 0.02. That
tolerance had been set from runs that never refitted a fine-tune. An amendment filed before anything else ran set it
to 2% of the gain. Every later number is a share of each state's own gain, so the offset does not enter it.

### Does removing the recruited set from the fine-tuned weights undo the fine-tune?

![recruitment](../experiments/recruitment_test/figures/recruitment.png)

**Reading it.** Partly on two tasks, barely on the third, and the filed pass was met nowhere. The blue bars reach 0.50
and 0.54 on emoticon, 0.32 and 0.36 on ` which`, and about 0.2 on ` we`. The random sets, grey dots, average near zero
everywhere. So on the word tasks S's share is well clear of chance, and the two seeds agree. On emoticon one random draw
reaches 0.26 on both seeds, with heavy damage elsewhere. That single draw breaks the filed rule, which scores no task
whose worst random set passes 0.15, so emoticon is unscored. The null's recruited set, orange, removes at most
0.13 of the gain. The busy set removes more than the whole gain and wrecks the model.

| task | share removed by S, seeds 0 / 1 | random mean | null's set | filed class |
|---|---|---|---|---|
| emoticon | 0.50 / 0.54 | 0.03 | 0.09-0.13 | unscored: a random set bites |
| ` which` → ` was` | 0.32 / 0.36 | 0.00 | 0.04-0.05 | partial |
| ` we` → ` just` | 0.17 / 0.22 | 0.00 | 0.07 | seeds disagree |

In [1]:
# The stage-1 numbers behind the first two figures, read from the results file (no model).
import json
from pathlib import Path

st = json.loads(Path("../experiments/recruitment_test/results/stage1.json").read_text())["states"]
print(f"{'state':18} {'share S':>8} {'share S at W0':>14} {'random mean':>12} {'null set':>9} {'busy':>6}")
for k, r in st.items():
    print(f"{k:18} {r['rho_S']:8.3f} {r['rho_S_W0']:14.3f} {r['random_mean']:12.3f} {r['rho_null']:9.3f} {r['rho']['busy']:6.2f}")

state               share S  share S at W0  random mean  null set   busy
emoticon/s0           0.503          0.258        0.031     0.131   1.60
emoticon/s1           0.543          0.267        0.034     0.091   1.61
r1_t534_v369/s0       0.323          0.160       -0.012     0.055   2.18
r1_t534_v369/s1       0.361          0.135        0.001     0.043   2.18
r2_t359_v816/s0       0.169          0.243       -0.006     0.067   1.44
r2_t359_v816/s1       0.216          0.254        0.001     0.071   1.47


### Is the recruited set inert in the original model?

![w0](../experiments/recruitment_test/figures/w0.png)

**Reading it.** No. This reverses a prediction we filed with high confidence. The hollow green circles are S removed
from the original weights; we expected them inside the grey band. Every one sits above it, at 0.13 to 0.27 of the gain,
a fall of 1.5 to 3.6 nats in the target's log-odds. On ` we` removing S lowers the target more in the original model
than in the fine-tuned one.

The independent check found no code fault; the miss is in the design. S is chosen by a *rise* in importance at the
trigger, with no requirement that it was off before. The removal also acts at every position, while S is defined at
the trigger only. A typical member of S is on at about 2% of ordinary Pile positions. So "S is inert in the original
model" was never what this measurement tests.

**An exploratory reading, not filed.** Removing S takes the original subcomponents out of both models, while the
fine-tune's own weight change stays. The gap between the two removals, printed above each pair, is then the share of
the gain lost when S is missing from both models. It is about 0.25 on emoticon, about 0.2 on ` which`, and slightly
negative on ` we`. Read this way, the fine-tune keeps about three quarters of its gain or more without S. That reading
needs random sets removed from the original weights before it means anything.

### Is removing the recruited set confined to the trigger?

![offtarget](../experiments/recruitment_test/figures/offtarget.png)

**Reading it.** No, except on ` we`. Away from the trigger, removing S, the blue dots, costs 17 to 22 times the
fine-tune's own damage on emoticon and 6 to 7 times on ` which`. Only on ` we` does it stay under the filed bar of three
times. That rule was not scored, because recruitment passed nowhere; read anyway, it fails on two tasks. Removing S
from the original model costs about the same. Still, the target effect of S is not general damage. On ` which` the
null's set costs more off target than S, yet removes a twentieth of the gain against S's third.

### Does removing more of the most-risen subcomponents undo more?

![topk](../experiments/recruitment_test/figures/topk.png)

**Reading it.** Yes, but beyond the recruited set this is broad damage. On the word tasks the share passes 1 by 100
to 200 subcomponents, with off-target damage of several nats. Past that size the added subcomponents barely rose, and
no random sets matched to each size were run. The curve says nothing about where the rest of the fine-tune lives.

### Does the fine-tune move the trigger's inputs toward where the original model predicts the target?

![routing](../experiments/recruitment_test/figures/routing.png)

**Reading it.** The direction is aligned, but the filed rule held only on ` which`, and its null bar did no work. The
blue markers sit above zero almost everywhere after layer 0, typically 0.1 to 0.4, far above a random direction's 0.04.
In layer 3, which holds most of S, ` which` passes: seed 0 reaches 0.3 on all four inputs, seed 1 on three.
` we` reaches it only on the MLP hidden input and misses. Emoticon would pass but is unscored. The null's cosines in layer 3 are all negative, so the
"three times the null" bar was met automatically. The query, key and value matrices share one input, so three of the
six layer-3 slots are one measurement; counting distinct inputs changes no verdict.

One reading runs against the "reveal" picture. The inputs end farther from the target's mean input, not nearer, and
farther still from the overall mean. The fine-tune pushes the trigger's inputs outward, partly along the target's
direction.

### What are the recruited subcomponents in the original model? (provisional)

![census](../experiments/recruitment_test/figures/census.png)

**Reading it.** The census's filed check has no power. It asks whether the target is among each subcomponent's top three
next tokens, ranked by over-representation. That ratio is capped, and rare tokens seen five times hit the cap. So almost
every subcomponent, in S or in a random set, reads "neither", and the random sets' target share is zero. What the
figure does show: members of S are on twice as often in ordinary text as random ones. The median member of S is on
at about 2.3% of positions, the median random one at 1.2%. They are common workers, not rare specialists.

Read by eye, and provisional: on ` which` many members of S are on after a noun phrase and predict a verb next.
On ` we` they are on after pronouns and predict verbs or contractions. S may be "a verb comes next" machinery. Two emoticon
members do favour the target `o` strongly.

## What we learned

The fine-tuned model does use the switched-on subcomponents for the target: removing them lowers it far more than
random or null sets on the two word tasks. So the switched-on set is not only Γ's reading. But the original model used
the same subcomponents for the target before the fine-tune, and removing them damages more than the trigger. This run
cannot yet tell "the fine-tune works by switching these on" from "they always pushed toward the target, and the
fine-tune works mostly elsewhere". For the "reveal" hypothesis in `GOALS.md`, that is weak support at best.

## How compelling is the evidence

- Three tasks, two seeds each, one null per task, one 67M model, held-out windows, the trigger position only.
- The share against random sets is solid on the word tasks; on emoticon one damaging random draw spoils the filed bar.
- Shares are in log-odds at near-certainty, so the effect in probability is much smaller.
- Removal acts at every position, so S's use elsewhere enters every number.
- The input trace's null bar was vacuous; the census's check had no power.

## Surprises

**Something unexpected happened that I cannot explain.** We predicted with high confidence that removing the recruited
set from the original model would do nothing. Instead it lowers the target by 0.13 to 0.27 of the fine-tune's gain on
all six states. The design explains how the prediction could fail, but not what S does at the trigger in the original
model. Also unexpected: the fine-tune moves the trigger's inputs away from the target's mean input, while pointing
partly along the target's direction.

## What next

1. **Remove random and null sets from the original model.** If S's gap between the two models stays well above theirs,
   that gap is the fine-tune's own use of S. If theirs is similar, the reversal is generic removal damage.
2. **Remove S at the trigger position only.** If the original-model share falls to near zero, the reversal came from S's
   use elsewhere in the window.
3. **Re-run the census saving its full counts,** and read the target's own enrichment per subcomponent against random
   sets.

## Scorecard

| id | claim | read | verdict |
|---|---|---|---|
| stage 0 | apparatus reproduces the earlier run | all checks within tolerance, after amendment 1 | pass, run 2 |
| P1 | removing S undoes ≥ 0.5 of the gain on two tasks | emoticon unscored; ` which` partial; ` we` seeds disagree | miss; tree "outside" |
| P2_S | removing S costs ≤ 3 × the fine-tune's off-target KL | 22×/17×, 6×/7×, 2×/2× | unscored; would fail on two |
| P3 | S is inert in the original model, \|share\| ≤ 0.1 | 0.13 to 0.27 on all six | **reversed** |
| P4 | layer-3 cosines ≥ 0.3 and ≥ 3 × null | ` which` pass, ` we` miss, emoticon unscored | miss; null bar vacuous |
| census | target share in S ≥ 3 × random | 0.005 vs 0, 0 vs 0, 0 vs 0 | no power |

## Not run

- Random and null sets removed from the original weights.
- Removal at the trigger position only.
- Random sets matched to each top-k size.
- The target's own enrichment per subcomponent: the census counts were not saved.
- The input trace's positive control, and other tokens' natural directions.
- Seeds beyond two, other tasks, the larger model.

Provenance: RunPod RTX 4090, about $0.23 for both stage-0 runs and stages 1 to 3; results in
`experiments/recruitment_test/results/stage{0,1,2,3}.json`, logs beside them; figures from `plot.py`.

# Amendment 2: does the fine-tune lean on the recruited set more than the original model did?

*2026-10-27. Plan `reports/plans/2026-10-27_plan_e38-amend2.md`; analysis
`reports/analyses/2026-10-27_analysis_e38-amend2.md`; independent check
`reports/reviews/2026-10-27_review_e38-amend2-check.md`; figures from `experiments/recruitment_test/plot_amend2.py`.*

## The answer

Yes by the filed rule, but the controls leave it open. Without the recruited set, the fine-tuned model loses a tenth to
a quarter more of its gain than the original model does, on two of three tasks. No random set comes close. But the set
switched on by a fine-tune to an unrelated token shows an extra loss about half as large, and equal on one seed. So the
fine-tune may simply amplify whatever is active at the trigger. Most recruited subcomponents already predict the target
in the original model.

## What we were trying to test

The first pass removed the recruited set S from the weights at every position. It took a third to a half of the gain
from the fine-tuned model on two tasks, but already a sixth to a quarter from the original model. So it could not say
how much of the fine-tune itself works through S. This amendment removes each set from *both* models, and adds a second
form of removal: S's rank-1 outputs are subtracted at the trigger position alone, every other position unchanged.

For a set X, with the gain $G = T_1 - T_0$ in the target's log-odds at the trigger:

- $\rho_{ft}(X)$ is the share of $G$ lost when X is removed from the fine-tuned weights;
- $\rho_0(X)$ is the share lost when X is removed from the original weights, over the same $G$;
- the *difference* $D(X) = \rho_{ft}(X) - \rho_0(X)$.

D is 0 when X matters to the target as much after the fine-tune as before, and 1 when the whole gain needs X present.
It reads how much of the gain needs X, not whether the gain flows through X's own outputs.

Three references. **Twenty random sets** of S's size per matrix rule out that removing that many subcomponents costs the
fine-tuned model more whichever they are. **The null's set** is the one switched on by a fine-tune to ` http` at the same
triggers. It rules out that any set a fine-tune here switches on shows the same difference. Its size is not matched to S. **The busy set**, the subcomponents most important at the trigger in the original model, checks that
removal at the trigger bites.

A **census** over 2,000 Pile rows asks what S is in the original model. A subcomponent is *target-enriched* when the
target follows the positions where it is on at least three times as often as it follows positions in general, with
at least five such cases. The control is the set most on before the target in the original model.

**What we expected.** On emoticon and ` which`, D between 0.10 and 0.35 and above every random set; on ` we`, not.
The null's set within ±0.10 on every state. S pushing toward the target at the trigger in the original model, and
target-enriched on emoticon and ` which`.

**The apparatus held.** The new hook at every position reproduces the first pass's removal exactly. Removal at the
trigger leaves earlier positions identical, and the busy control moves the trigger by 11 nats. The states, regenerated
on an L40 instead of the first pass's 4090, differ by up to 0.12 nats in the fine-tuned log-odds. That moves the first
pass's shares by at most 0.006.

### Does the fine-tuned model lose more without S than the original model does?

![D](../experiments/recruitment_test/figures/amend2_D.png)

**Reading it.** Yes on emoticon and ` which`, more than any random set, and no on ` we`; but the null's set is not
where we expected it. The blue dots are D for S. Removed at the trigger only, the left panel, they read 0.23 and 0.25 on
emoticon and 0.12 and 0.19 on ` which`, above the dashed bar and above all twenty grey random dots. On ` we` they sit
just below zero. That is the filed "through" reading.

The orange diamonds are the ` http` fine-tune's set. They should sit in the grey band; instead they read 0.10 to 0.14
on emoticon and ` which`, and are positive on all six states. On ` which` seed 0 the null's set reads 0.113 against S's
0.116. The busy set, off the top, reads 0.78 to 1.10, more than S everywhere. So beating random sets is not special to
S: random sets are mostly off at the trigger, and any set active there may show a difference. Removal everywhere, the
right panel, tells the same story, with one heavy random draw on emoticon seed 1.

### Where does each set's difference come from?

![shares](../experiments/recruitment_test/figures/amend2_shares.png)

**Reading it.** S pushes toward the target at the trigger in the original model, and the null's difference arises
differently on each task. The hollow blue circles, S removed from the original weights at the trigger alone, read 0.18
to 0.27 on every state, above the filed bar. The filled ones, from the fine-tuned weights, sit higher on emoticon and
` which` and lower on ` we`. The null's set does nothing in the original model on emoticon, yet takes up to 0.14 of the
gain from the fine-tuned one. On ` which` removing it from the original model *raises* the target, by about 0.9 nats,
which makes most of its difference there.

On ` we` only 9 and 5% of S was below 0.1 importance at the trigger in the original model, against a filed 30
to 90%. There Γ turns up subcomponents that were already on, and D is negative.

### Are the recruited subcomponents the original model's machinery for the target?

![census](../experiments/recruitment_test/figures/amend2_census.png)

**Reading it.** Yes, on all three tasks. The blue bars, the target-enriched fraction of S in ordinary Pile text, read
0.60, 0.80 and 0.96. The dark grey random sets read 0.02 to 0.08, and the gold control 0.74 to 0.92. Counting each
random subcomponent once, the light grey bars, changes nothing to two decimals.

Two limits. The enrichment is only partly specific: the ` which` set is also enriched for ` just`, at 0.38, and the ` we`
set for ` was`, at 0.48, so the word tasks' sets look like "a verb or adverb comes next" machinery. And ` we`, with the
most enriched S, is the task with negative D. The census says what S is, not whether the fine-tune works through it.

## What Amendment 2 means

**The decision tree reads "through"** by its filed rule: emoticon and ` which` on both seeds. That branch writes D into
the experiment's card and Q5 and raises a card for the next step.

**The null and busy sets undercut it.** The alternative is that the fine-tune changes what happens downstream of the
trigger, and that change amplifies whatever the trigger feeds it. Then any set active at the trigger costs more in the
fine-tuned model. On that reading only S's excess over the null's set belongs to S: 0.09 and 0.16 on emoticon, 0.00
and 0.08 on ` which`, negative on ` we`.

**What would settle it, proposed and not run.** An independent check proposed this test on the cached states. In the
fine-tuned model, set S's outputs at the trigger back to their values in the original model, rather than removing them. That
undoes only S's switch-on. Do the same for the null's set minus S. A restored share of 0.1 or more for S and about 0
for the null says the gain runs through S's switch-on. Alike shares say the effect is generic. It is minutes on the
cluster and goes to the user as a pending card.

**What we learned.** The recruited subcomponents are the original model's own target machinery, and they already push
toward the target at the trigger. The fine-tuned model needs them more on two tasks. That fits `GOALS.md`'s
"strengthen" reading. Whether the fine-tune works through them, rather than through everything active at the trigger,
is still open.

**How compelling.** Three tasks, two seeds, one null each, one 67M model, the trigger position only. The apparatus is
exact. The weak point is the reference: no set matched to S on activity at the trigger was run, and the null's set is
not size-matched.

**Surprises.** The null's set, expected to show no difference, shows one on every state and matches S on one seed. On
` we`, S was mostly already on at the trigger in the original model. The census passed on ` we` too.

## Amendment 2 scorecard

| id | claim | read | verdict |
|---|---|---|---|
| C0-C4 | apparatus: states, hook, earlier positions, empty set, busy bites | 0.0 where 0 expected; busy KL 11.2 | pass |
| P5 | D(S) ≥ 0.10 and above all 20 random, trigger only, ≥ 2 tasks | emoticon 0.23, 0.25; ` which` 0.12, 0.19; ` we` −0.06, −0.02 | pass: "through" |
| P5_all | same classes, removal everywhere | emoticon seeds disagree | miss |
| P5_null | \|D(null)\| ≤ 0.10, every state | 0.14, 0.10, 0.11, 0.11, 0.06, 0.07 | **fail** on four |
| P6 | ρ0(S) ≥ 0.10 at the trigger, every state | 0.18 to 0.27 | pass |
| P7 | off-target KL ≤ 3 × the fine-tune's, ≥ 2 tasks | ratio 0.01 to 0.04 | pass |
| P8 | target-enriched fraction ≥ 2 × random and ≥ 0.10, ≥ 2 tasks | 0.60, 0.80, 0.96 vs 0.07, 0.08, 0.02 | pass, all three |
| reported | 30-90% of S below 0.1 importance in W0 | 83%, 84%, 55%, 57%, 9%, 5% | miss on ` we` |

## Amendment 2: not run

- Restoring S's trigger outputs to their original-model values in the fine-tuned model; the same for the null's set
  minus S.
- Random sets matched to S on activity at the trigger, or to the null's set in size.
- The census against other tokens of the same class.

Provenance: MATS cluster L40, jobs 38805 and 38809, about two minutes, $0; results in
`experiments/recruitment_test/results/amend2{_stage0,,_census}.json`, logs beside them.

# Amendment 3: does the gain run through the recruited set's changed outputs?

*2026-10-27. Plan `reports/plans/2026-10-27_plan_e38-amend3.md`; analysis
`reports/analyses/2026-10-27_analysis_e38-amend3.md`; independent check
`reports/reviews/2026-10-27_review_e38-amend3-check.md`; figures from `experiments/recruitment_test/plot_amend3.py`.*

## The answer

Partly yes, against what we predicted. Putting the recruited subcomponents' trigger outputs back to their original
values takes back about a fifth of the gain on emoticon and ` which`, and a twentieth on ` we`. Every other set we
tried takes back about nothing. The set switched on by an unrelated fine-tune, which removal had said carried a tenth
of the gain, carries none. The filed rule reads "outside the tree": ` which` is specific, ` we` too small, emoticon
unscored.

## What we were trying to test

Amendment 2 removed the recruited set S at the trigger and found the fine-tuned model lost more without it than the
original model did. But the set switched on by a fine-tune to ` http`, the *null's set*, showed half that extra loss,
and the busy set more. Removal takes out a set's whole output, including what it did before the fine-tune, so it could
not tell whether S carries the change or the fine-tuned model just leans harder on whatever is present.

This amendment *restores* instead. A subcomponent's output at the trigger is $(x \cdot v_c)\, u_c$, with $x$ its
matrix's input there. Its weights do not change, so its output changes only because its input does. Restoring a set X
in the fine-tuned model adds $\sum_{c \in X} ((x_0 - x) \cdot v_c)\, u_c$ to each matrix's output at the trigger
only, with $x_0$ the input in the original model's pass on the same window. The weight change's own output stays. The
*restored share* is the drop in the target's log-odds over the fine-tune's gain: 0 when the set's changed outputs
carry nothing, 1 when undoing them takes the target back to the original model's value.

Four kinds of set, per task and seed. S itself. Twenty random sets of S's size, to rule out that any that many
subcomponents carry the change. The null's set minus S, to test Amendment 2's confound. And twenty *active* sets: per
layer, S's count drawn from the other subcomponents already important at the trigger in the original model, at least
0.5 or, where that was short, 0.2. They rule out that any busy subcomponents carry as much as S.

**What we expected.** S would beat random sets on emoticon and ` which`, medium confidence. But S would sit inside the
active sets' range on at least two tasks, low confidence, because the busy and null sets had cost so much under removal.

**The apparatus held, on the second run.** Four controls read 0.0 on all nine states, and putting back every matrix's
whole trigger output takes back the whole gain. The fifth control compares restoring with $x_0 = 0$ against Amendment
2's removal. The first run stopped there: ` we` seed 1 differed by 0.0195 nats against a bar of 0.01, set without
measuring bf16 rounding. We filed a looser bar before any restored share had been computed, and the rerun's control
numbers matched to the digit. A sign error would read about 5 nats.

In [2]:
# The restored shares behind the figures, read from the results file (no model).
import json
from pathlib import Path

a3 = json.loads(Path("../experiments/recruitment_test/results/amend3.json").read_text())
print(f"{'state':18} {'S':>7} {'null minus S':>13} {'active max':>11} {'random max':>11}")
for k, v in a3["states"].items():
    r = v["rho"]
    print(f"{k:18} {r['S']:7.3f} {r['null_minus_S']:13.3f} {max(r['active']):11.3f} {max(r['random']):11.3f}")
print("P9 reading:", a3["P9"]["reading"], "| P10 pass:", a3["P10"]["pass"])

state                    S  null minus S  active max  random max
emoticon/s0          0.239        -0.024      -0.186       0.029
emoticon/s1          0.220        -0.039      -0.160       0.017
r1_t534_v369/s0      0.179         0.001       0.041       0.021
r1_t534_v369/s1      0.202         0.016       0.047       0.012
r2_t359_v816/s0      0.037         0.010       0.032       0.007
r2_t359_v816/s1      0.049         0.022       0.034       0.009
P9 reading: outside the tree | P10 pass: True


### Does putting S's trigger outputs back take more of the gain than other sets?

![restore](../experiments/recruitment_test/figures/amend3_restore.png)

**Reading it.** Yes on emoticon and ` which`, barely on ` we`. The blue dots, S, read 0.18 to 0.24 on emoticon and
` which`, above the dashed bar, and 0.04 and 0.05 on ` we`, under the dotted one. The grey random sets and the orange
null's set sit at zero everywhere. The violet active sets sit near zero on the word tasks, at most 0.05. On emoticon
they sit near −0.2: putting them back *raises* the target.

On ` we` S is above every active set too, so it reads "not" because its share is too small to matter. One caveat, from
the independent check. On ` which` and emoticon, a layer's pool was barely larger than S's count there, so the twenty
draws overlap heavily. On ` which` seed 1 every draw holds the same 44 layer-3 subcomponents. Beating all twenty is
closer to beating one set with jitter.

### Does the null's set still carry part of the gain once only its change is undone?

![confound](../experiments/recruitment_test/figures/amend3_confound.png)

**Reading it.** No: the restore test resolves Amendment 2's confound. Under removal, the left panel, the null's set read
0.10 to 0.14 on emoticon and ` which`, as high as S on one seed. Under restore, the right panel, it reads −0.04 to
+0.02 on every state, inside the random range, while S keeps a fifth of the gain. So the null's set did not change its
outputs in a way that carries the gain; the fine-tuned model just leans more on what it already outputs. The active
sets, drawn from the same kind of busy subcomponents as Amendment 2's busy set, restore zero or less.

### Why is emoticon unscored, and what would it read?

![pool](../experiments/recruitment_test/figures/amend3_pool.png)

**Reading it.** The original model has too few active subcomponents at the emoticon trigger to match S. In layer 3, S
has 127 members and the pool at 0.2 holds 107, so each draw is 20 short. The pool rule, filed before the run, leaves
such a state unscored, since smaller sets favour S.

*Exploratory, outside the filed rule:* scored as they are, the short draws would give emoticon "specific" on both
seeds. S sits about 0.4 above every draw, and the missing 20 would come from barely active subcomponents, which take
back nothing in random sets. The overall reading would then be "specific" on two tasks.

*Provisional, not filed:* why are the active shares negative on emoticon? Their outputs change only through their
inputs, so the fine-tune's change to the trigger's inputs makes the original model's active subcomponents push against
` o`, by 2.7 to 3.0 nats. The gain comes despite them. Either this is spread over the active machinery as a side effect,
or a few layer-3 subcomponents carry it. Single random draws reach −0.13, and every active draw holds the whole layer-3
pool, which would explain the tight spread. Restoring that pool alone would tell.

## What Amendment 3 means

**The decision tree reads "outside the tree"**: ` which` "specific", ` we` "not", emoticon unscored. Its branch is a
pending card with the numbers and no further runs. The strongest open alternative, from the independent check: S was
chosen by a rise in importance, which tracks how much its outputs changed, and no comparison set was matched on that
change. Proposed options for the card:

- **(a) Restore the subcomponents whose trigger outputs changed most**, outside S and matched to S per layer, plus
  layer 3's active pool alone. Minutes on the cluster, $0, cached states. If S still wins, "specific" holds against the
  strongest alternative and emoticon is scored; if not, S is just the set whose outputs moved most. Recommended.
- **(b) Accept "specific on two tasks, provisional"** and write it into Q5.
- **(c) Stop the re-gating line on the 67M model**, or carry the restore test to the Qwen line.

**What we learned.** The subcomponents that Γ switches on carry part of the fine-tune's change themselves, about a
fifth of the gain on two tasks, and no other tested set carries any. Amendment 2's worry is real but separate: the
fine-tuned model leans more on what is present at the trigger, which shows under removal and vanishes under restore.
With the census, which found S to be the original model's own target machinery, this fits the "strengthen" reading.

**How compelling.** Three tasks, two seeds, one 67M model, the trigger only. The controls are exact. Specific on one
task by the filed rule, two in the exploratory reading. The active draws overlap, and no set is matched on output
change. Restored shares need not add.

**Surprises.** Something we did not expect: on ` which`, S is specific among the active subcomponents, reversing the
filed prediction. We predicted, with low confidence, that it would sit inside their range; it sits 0.14 and 0.16 above
the highest. On emoticon, the original model's active subcomponents push against the target after the fine-tune. The
null's set, a tenth of the gain under removal, carries none under restore.

## Amendment 3 scorecard

| id | claim | read | verdict |
|---|---|---|---|
| R0-R3 | restore nothing; S in the original model; whole output; earlier positions | 0.0 on all nine states | pass |
| R4 | restore with x0 := 0 matches removal | 0.0002-0.0195 nats, at most 0.7% of the removal | first run fail at 0.01; pass at the amended bar |
| P9 | S specific among active sets, ≥ 2 tasks | ` which` 0.18, 0.20 vs ≤ 0.05; ` we` 0.04, 0.05; emoticon pool short | **outside the tree**; "not specific" reversed on ` which` |
| P10 | S ≥ 0.10 and above all 20 random, emoticon and ` which` | 0.18 to 0.24 vs ≤ 0.03 | pass |
| reported | S's share 0.10-0.50, ` we` below 0.10; pool at 0.5 | as expected; pool at 0.5 on ` we` only | pass; miss on four |

## Amendment 3: not run

- Comparison sets matched on how much their trigger outputs changed.
- Layer 3's active pool alone on emoticon, or its members one at a time.
- Restoring S together with an active set, or every alive subcomponent at once.

Provenance: MATS cluster L40, jobs 38950, stopped at the controls, and 38953, about 40 s each, $0; results in
`experiments/recruitment_test/results/amend3.json`, the stopped run in `amend3_run1.json`, logs beside them.

# Amendment 4: is the recruited set special, or just the set whose outputs changed most?

*2026-10-28. Plan `reports/plans/2026-10-28_plan_e38-amend4.md`; analysis
`reports/analyses/2026-10-28_analysis_e38-amend4.md`; independent check
`reports/reviews/2026-10-28_review_e38-amend4-check.md`; figures from `experiments/recruitment_test/plot_amend4.py`.*

## The answer

Yes, against this alternative. Putting the recruited set's trigger outputs back takes back about a fifth of the gain
on emoticon and ` which`. The same number of subcomponents whose outputs moved most carry four times the change, yet
take back nothing on ` which` and less than nothing on emoticon. Against what we expected, the recruited set is not
where the outputs moved most. The filed rule reads "specific"; the independent check agrees. Whether the set's changes
merely point toward the target is untested.

## What we were trying to test

Amendment 3 restored sets: in the fine-tuned model, at the trigger only, it put a set's outputs back to their values
in the original model. A subcomponent's weights do not change, so its output changes only because its input does.
Restoring the recruited set S took back about a fifth of the gain on emoticon and ` which`. Random, null and
already-active sets took back about nothing. But S was picked by a rise in importance, and that rise might just track
how far a subcomponent's input moved. Then S would simply be the set whose outputs changed most.

So this amendment measures each subcomponent's *output change* at the trigger,
$\Delta_c = |(x_1 - x_0) \cdot v_c|\, \lVert u_c \rVert$, averaged over the training windows. Here $x_1$ and $x_0$
are the matrix's trigger inputs in the fine-tuned and original models, and $u_c v_c^\top$ is the subcomponent. It is
the size of the correction that restoring $c$ adds, and it is blind to direction. Two sets, built per layer from the
alive subcomponents outside S, are then restored on the held-out windows:

- the **count-matched set**: as many as S has in the layer, those with the largest change;
- the **change-matched set**: the largest movers, in order, until their summed change equals S's in the layer.

They rule out that any set that moved as much carries as much. Restoring **every alive subcomponent** at once shows
how much of the gain the subcomponents' changed outputs carry at all. On emoticon, to explain Amendment 3's negative
share, the layer-3 active pool is restored together, then each of its 107 members alone, beside as many random
layer-3 subcomponents alone.

**What we expected.** S would take back at least 0.05 more than both matched sets on emoticon and ` which`, low
confidence. S would hold the top movers. Emoticon's negative share would sit in layer 3, in a few subcomponents.

**The apparatus held.** Amendment 3's five controls reproduced to the digit, and S's share matched Amendment 3's to
four decimals. A new control, the output change with the fine-tuned weights set to the original ones, read exactly 0.
That control tests only that the passes are deterministic.

### Does S take back more of the gain than the subcomponents whose outputs changed as much or more?

![restore](../experiments/recruitment_test/figures/amend4_restore.png)

**Reading it.** Yes, on emoticon and ` which`. The blue dots, S, read 0.18 to 0.24. The green squares, the
count-matched sets, read +0.02 and +0.04 on ` which` and about −0.14 on emoticon. The red triangles, the
change-matched sets, read −0.02 to −0.04 on ` which` and about −0.23 on emoticon. They need only 8 to 28
subcomponents, the red numbers. On ` we`, S sits under the dashed floor, so that task is not scored.

` which` carries the claim. There S sits 0.16 and 0.17 above the count-matched set, about as far as it sits above zero,
with no negative block. On emoticon the margin is mostly overlap: the change-matched set's layer-3 members sit in the
layer-3 active pool, and its share equals that pool's. Against zero, S's emoticon share is the same fifth.

The black bars, every alive subcomponent at once, read 0.38 to 0.49. All their changed outputs at the trigger carry
under half of the gain. The rest sits in what the weight change writes there directly. S alone reads about two fifths
to three fifths of that.

### Is the recruited set the set whose trigger outputs changed most?

![change](../experiments/recruitment_test/figures/amend4_change.png)

**Reading it.** No, reversing what we expected. In the top panel, the green count-matched sets carry 3.3 to 4.3 times
S's output change with the same number of subcomponents. The red change-matched sets reach S's change with 6 to 28
subcomponents, against S's 21 to 213. In the bottom panel, S makes up 5% to 11% of the top movers per layer, against a filed half. A random set
of S's size would read under 3%, so S is over-represented, but it is not the top movers.

Taken with the figure above: the subcomponents whose outputs moved most carry none of the gain, or work against it,
while S's smaller changes take back a fifth. How far an output moved does not predict how much of the gain it carries.
The change measure is a size, so whether S stands out because its changes point toward the target is untested. A
ranking without Γ, by each subcomponent's first-order effect on the target logit, might find a set that does as well.
Matching per layer also mixes query, key, MLP and residual outputs, and the matched sets' matrices were not saved.

### On emoticon, does the negative share of the active subcomponents sit in a few of them?

![singles](../experiments/recruitment_test/figures/amend4_singles.png)

**Reading it.** Yes. The layer-3 active pool restored together reads −0.22 on both seeds; the rest of an active set
reads about zero. Restored one at a time, almost every pool member reads within 0.02 of zero. The exceptions are the
same three layer-3 MLP subcomponents on both seeds. Two in the MLP's output matrix read −0.10 to −0.135 each, about a
tenth of the gain against the target, and one in its input matrix about −0.07. Random layer-3 subcomponents read
within 0.003 of zero. Restored shares need not add, so how the block's −0.22 divides among the three is not known. Why
they push against the target after the fine-tune is not measured.

## What Amendment 4 means

**The decision tree reads "specific"**: emoticon and ` which` on both seeds, ` we` too small to score. Its branch
writes a checked claim into the E38 card and Q5, in the wording the independent check allows:

> On the 67M model after a 10-step fine-tune, putting back the original trigger outputs of the subcomponents Γ
> switched on takes back about a fifth of the gain on emoticon and ` which`. The same number of the subcomponents whose
> trigger outputs changed most carry about four times the change, yet take back about none on ` which` and less than
> none on emoticon. So how much a subcomponent's output moved does not explain the switched-on set's share.

Not claimable: that Γ is needed to find these subcomponents, that emoticon's margin measures specificity, or anything
about ` we`.

**What we learned.** The share the recruited set carries is not explained by how far its outputs moved. This closes
the size half of the alternative Amendment 3 left open, and with the census it fits the "strengthen" reading. The
subcomponents' changed outputs together carry under half of the gain at the trigger.

**How compelling.** Three tasks, two seeds, one 67M model, the trigger only. Every verdict was recomputed by the
independent check. ` which` carries the claim; emoticon's margin is overlap with a negative block. Matching was on
size, per layer; direction and per-matrix matching are untested.

**Surprises.** Something we did not expect: S is not the set whose outputs moved most. We expected the count-matched
set to carry less change than S; it carries 3.3 to 4.3 times as much. We expected S to make up half the top movers; it
makes up a twentieth to a tenth. And single layer-3 MLP subcomponents each carry a tenth of the gain against the
target on emoticon.

**What next.** First, beside the upstream step, sets matched on direction, per matrix: rank by each subcomponent's
first-order effect on the target logit, with no Γ. If such a set takes back as much as S, Γ is not needed to find it.
Then the upstream step the tree names: which earlier changes moved the trigger's inputs so that S switches on.

## Amendment 4 scorecard

| id | claim | read | verdict |
|---|---|---|---|
| R0-R5 | Amendment 3's five controls; output change 0 with W1 := W0 | 0.0 on all states; R4 at most 0.7% of the removal | pass |
| P11 | S ≥ both matched sets + 0.05, on ≥ 2 scored tasks | emoticon 0.24, 0.22 vs ≤ −0.13; ` which` 0.18, 0.20 vs ≤ 0.04; ` we` S < 0.10 | **specific**, as expected |
| reported | count-matched change below S's, all six | 3.27 to 4.34 times S's | **reversed** on all six |
| reported | S ≥ half the top movers, emoticon and ` which` | 0.05 to 0.11 | **reversed** on all four |
| reported | every alive subcomponent between 0.3 and 0.9 | 0.38 to 0.49 | as expected |
| reported | layer-3 pool ≤ −0.15, rest within 0.05 | −0.22, −0.22; −0.03, −0.01 | as expected |
| P12 | most negative single ≤ −0.05 and 0.02 below random singles | −0.135, −0.115 vs −0.002 | **few**, as expected |

## Amendment 4: not run

- Sets matched on direction: a ranking without Γ by first-order effect on the target logit.
- Matching per matrix; which subcomponents and matrices the matched sets hold.
- The change-matched set on emoticon without the layer-3 pool.
- A control showing that a large output change gives a large restore.
- The upstream step; other positions, tasks, seeds, the larger model.

Provenance: MATS cluster `debug`, job 40041, L40, 63 s, $0, resubmitted twice at lower memory requests with no code
change; results in `experiments/recruitment_test/results/amend4.json` and `run_amend4.log`.